# Imports

In [1]:
import os
import pickle
import re
import shutil
import sys
sys.path.append(os.path.dirname(os.getcwd()))
from itertools import product

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from matplotlib.backends.backend_pdf import PdfPages
from tools import load_npy, load_yaml_as_df, load_pkl, exist_metric, exist_stf_metric, inverse_stf_metrics, keep_split, is_full_group, exist_pred

plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
plt.rc('font', family='Arial')
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.size'] = 10

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Snapshots

## load config

In [2]:
save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
baselines = pd.read_csv(f'{save_root}/baselines_params.csv')
finetunes_best = pd.read_csv(f'{save_root}/best_finetune_full_each.csv')

model_list = ['Fredformer', 'iTransformer', 'FreTS', 'DLinear']

best = finetunes_best.copy()
best = best[best.pred_len != 'Avg']
best = best[
    ((best.data_id == 'ETTm1_PCA') & (best.model.isin(['Fredformer']))) |
    ((best.data_id == 'ETTh1_PCA') & (best.model.isin(['Fredformer']))) |
    ((best.data_id == 'ECL_PCA') & (best.model.isin(['iTransformer']))) |
    ((best.data_id == 'Weather_PCA') & (best.model.isin(['FreTS'])))
]


In [ ]:
for i, row in best.iterrows():
    print(row['data_id'], row['pred_len'])
    exp_dir = row['exp_dir']
    print(exp_dir)
    result_dir = f'{exp_dir}/results'
    results = os.listdir(result_dir)
    result = os.path.join(result_dir, results[0])
    print(os.path.exists(os.path.join(result, 'pca_components.npy')))
    print('\n')

In [ ]:
columns = ['model', 'data_id', 'pred_len', 'mse', 'mae']

# df_show = base[columns].copy()
df_show = best[columns].copy()
df_show['pred_len'] = df_show['pred_len'].astype(int)
df_show.sort_values(by=['data_id', 'model', 'pred_len'], inplace=True)
df_show

## load data

In [3]:
data = {}

print("process finetunes ...\n\n")
for i, row in best.iterrows():
    exp_dir = row['exp_dir']
    runned, setting_dir = exist_pred(exp_dir)
    if not runned:
        print(os.path.basename(exp_dir), 'has no prediction!!')
        continue
    else:
        print(os.path.basename(exp_dir), ' ---> processing ...')

    try:
        model, data_id, pred_len = row['model'], row['data_id'], row['pred_len']
        inputs = load_npy(os.path.join(setting_dir, 'input.npy'))[..., -1]
        preds = load_npy(os.path.join(setting_dir, 'pred.npy'))[..., -1]
        trues = load_npy(os.path.join(setting_dir, 'true.npy'))[..., -1]
        pca_components = load_npy(os.path.join(setting_dir, 'pca_components.npy'))
        data[f"{model}_{data_id}_{pred_len}_Base"] = pca_components[-1]

        gt_ = np.concatenate([inputs, trues], axis=1)
        pd_ = np.concatenate([inputs, preds], axis=1)

        del inputs, preds, trues
        data[f"{model}_{data_id}_{pred_len}"] = [gt_, pd_]
    except Exception as e:
        print(f"Error processing {os.path.basename(exp_dir)}: {e}")
        continue

process finetunes ...


Fredformer_ETTm1_PCA_96_0.6_0.4_0.0005_TST_100_10_128_128_2_8_96_32_MAE_0_1_T_1.0  ---> processing ...
Fredformer_ETTm1_PCA_192_0.2_0.8_0.0005_TST_100_10_128_128_2_8_96_32_MAE_0_1_T_1.0  ---> processing ...
Fredformer_ETTm1_PCA_336_0.1_0.9_0.0005_TST_100_10_128_128_2_8_96_32_MAE_0_1_T_0.7  ---> processing ...
Fredformer_ETTm1_PCA_720_0.0_1.0_0.0005_TST_100_10_128_164_2_8_96_32_MAE_0_1_T_0.8  ---> processing ...
Fredformer_ETTh1_PCA_96_0.0_1.0_0.0005_type3_100_10_128_128_2_8_96_32_MAE_0_1_T_1.0  ---> processing ...
Fredformer_ETTh1_PCA_192_0.0_1.0_0.0005_type3_100_10_128_128_2_8_96_32_MAE_0_1_T_1.0  ---> processing ...
Fredformer_ETTh1_PCA_336_0.0_1.0_0.0005_type3_100_10_128_128_2_8_96_32_MAE_0_1_T_1.0  ---> processing ...
Fredformer_ETTh1_PCA_720_0.0_1.0_0.0005_type3_100_10_128_128_2_8_96_32_MAE_0_1_T_0.9  ---> processing ...
iTransformer_ECL_PCA_96_0.2_0.8_0.001_type1_10_3_16_MAE_0_1_T_1.0  ---> processing ...
iTransformer_ECL_PCA_192_0.3_0.7_0.001_type1_10_3_1

In [ ]:
for k, v in data.items():
    if isinstance(v, list):
        print(k, v[0].shape, v[1].shape)
    else:
        print(k, v.shape)

## plot

In [6]:
def plot(dataset, model, index, _save_dir, figure_name='predict', ypad=0, loc=None, limit=None, dpi=400, ylim=None):
    legend_location = 'upper left' if loc is None else loc
    labelsize = 12

    style = ['darkgrid', 'dark', 'white', 'whitegrid', 'ticks']
    context = ['paper', 'notebook', 'talk', 'poster']
    palette = sns.color_palette('deep')
    palette = [palette[3], palette[0], palette[2]] + palette[4:]
    sns.set_theme(style=style[0], context=context[0], font='Arial', font_scale=1.6, palette=[palette[0], palette[3], palette[2]] + palette[4:])

    f = plt.figure(dpi=dpi, figsize=(4, 3))  # 如果只画一条线，就用红色，高改为3->2.5
    f.subplots_adjust(top=0.9, left=0.1, right=0.9, bottom=0.2)

    f.add_subplot(1, 1, 1)
    true, preds = dataset[0][index], dataset[1][index]
    if limit is not None:
        true, preds = true[:limit], preds[:limit]

    plt.plot(true, label='GroundTruth', linewidth=2, color='black', zorder=2)
    plt.plot(preds, label='Prediction', linewidth=2, color=palette[0], zorder=1)
    ax = plt.gca()
    if ypad:
        ax.yaxis.set_major_locator(plt.MultipleLocator(ypad))  # 小数点后3位数间隔。
    if ylim:
        plt.ylim(ylim)
    plt.legend(ncol=1, loc=legend_location, fontsize=labelsize)
    # plt.title(model + ' & PDF', fontsize=14)
    plt.tick_params(labelsize=labelsize)

    plt.tight_layout()
    with PdfPages(os.path.join(_save_dir, f"{figure_name}_w_PDF.pdf")) as pdf:
        pdf.savefig(f, bbox_inches='tight', pad_inches=0.01)
    plt.close('all')


figure_root = '/data/home/Licheng/workspace/TSF-PCA/figure_PCA'
save_dir = os.path.join(figure_root, 'snapshots_rebb')
os.makedirs(save_dir, exist_ok=True)

DPI = 100

for model, data_id in [
    ('Fredformer', 'ETTm1_PCA'),
    ('Fredformer', 'ETTh1_PCA'),
    ('iTransformer', 'ECL_PCA'),
    ('FreTS', 'Weather_PCA')
]:
    for pl in [96, 192, 336, 720]:
        tag = f"{model}_{data_id}_{pl}"
        datap = data[tag]
        N = datap[0].shape[0]
        components = data[f"{tag}_Base"]
        datap2 = [np.dot(mat[:, -pl:], components.T) for mat in datap]

        save_dir_ = os.path.join(save_dir, data_id, str(pl))
        os.makedirs(save_dir_, exist_ok=True)
        for index in np.linspace(0, N - 1, 100):
            index = int(index)
            plot(datap2, model, index, save_dir_, figure_name=f'{index}_trans', loc='lower right', limit=40, dpi=DPI)


In [9]:
import subprocess

# 例：压缩 your_folder 到 output.zip
figure_root = '/data/home/Licheng/workspace/TSF-PCA/figure_PCA'
save_dir = os.path.join(figure_root, 'snapshots_rebb')
output_path = os.path.join(figure_root, 'snapshots.zip')
zip_folder = save_dir

if os.path.exists(output_path):
    os.remove(output_path)
subprocess.run(
    ['zip', '-r', output_path, os.path.basename(zip_folder)],
    cwd=figure_root,
    check=True
)

  adding: snapshots_rebb/ (stored 0%)
  adding: snapshots_rebb/ETTh1_PCA/ (stored 0%)
  adding: snapshots_rebb/ETTh1_PCA/96/ (stored 0%)
  adding: snapshots_rebb/ETTh1_PCA/96/1265_trans_w_PDF.pdf (deflated 6%)
  adding: snapshots_rebb/ETTh1_PCA/96/984_trans_w_PDF.pdf (deflated 6%)
  adding: snapshots_rebb/ETTh1_PCA/96/1406_trans_w_PDF.pdf (deflated 6%)
  adding: snapshots_rebb/ETTh1_PCA/96/1884_trans_w_PDF.pdf (deflated 6%)
  adding: snapshots_rebb/ETTh1_PCA/96/590_trans_w_PDF.pdf (deflated 6%)
  adding: snapshots_rebb/ETTh1_PCA/96/1996_trans_w_PDF.pdf (deflated 6%)
  adding: snapshots_rebb/ETTh1_PCA/96/956_trans_w_PDF.pdf (deflated 6%)
  adding: snapshots_rebb/ETTh1_PCA/96/449_trans_w_PDF.pdf (deflated 6%)
  adding: snapshots_rebb/ETTh1_PCA/96/731_trans_w_PDF.pdf (deflated 6%)
  adding: snapshots_rebb/ETTh1_PCA/96/337_trans_w_PDF.pdf (deflated 6%)
  adding: snapshots_rebb/ETTh1_PCA/96/309_trans_w_PDF.pdf (deflated 6%)
  adding: snapshots_rebb/ETTh1_PCA/96/1124_trans_w_PDF.pdf (deflate

CompletedProcess(args=['zip', '-r', '/data/home/Licheng/workspace/TSF-PCA/figure_PCA/snapshots.zip', 'snapshots_rebb'], returncode=0)